# STY prediction and interpretation

Optuna searches the original XGBoost parameter ranges using five training-set folds. The 80/20 split and 19 inputs match the model comparison. Calcination means are calculated within each fitting split. Cross-validation scores guide model selection. The held-out test set provides final evaluation.

In [ ]:
from pathlib import Path
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap
from IPython.display import display
from sklearn.model_selection import train_test_split

from concurrent.futures import ThreadPoolExecutor
import optuna
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold
from xgboost import XGBRegressor

from itertools import combinations
from sklearn.inspection import partial_dependence

In [ ]:

input_path = Path("derived") / "dataset.xlsx"  # Path to the input workbook.
output_dir = Path("STY_results")  # Folder for generated results.
output_dir.mkdir(exist_ok=True)
seed = 0  # Random seed for reproducibility.
dpi = 600  # Resolution in dots per inch.

mpl.rcParams.update({
    "font.family": ["Arial", "DejaVu Sans"],
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.0,
    "savefig.facecolor": "white",
})

features = [  
    "wt% Cu", "wt% Support1", "χ Support1", "wt% Support2", "χ Support2",
    "wt% Support3", "χ Support3", "wt% Promoter", "χ Promoter",
    "Synthesis Method", "Calc T", "Calc D", "H2/CO2", "GHSV", "P", "T",
    "SBET", "SACu", "dCu",
]
point_id = "Data point number"  # Unique experimental-record identifier column.
calc_columns = ["Calc T", "Calc D"]  # Calcination temperature and duration columns.
flag_columns = [f"{column} imputed" for column in calc_columns]  # Columns indicating missing calcination values. Current column name.
data = pd.read_excel(input_path, sheet_name="data", engine="openpyxl")  # Input records for this analysis.
if data[point_id].isna().any() or not data[point_id].is_unique:
    raise ValueError("Data point numbers must be present and unique.")
X = data[features].apply(pd.to_numeric).astype(float)  # Predictor matrix.
if not np.isfinite(X.to_numpy()).all():
    raise ValueError("Encoded model inputs must be finite.")
for column, flag in zip(calc_columns, flag_columns):  # Current column name; Indicator for the current record or column.
    if not data[flag].isin([False, True]).all():
        raise ValueError(f"{flag} must contain Boolean values.")
    X[column] = X[column].mask(data[flag].astype(bool))

train_idx, test_idx = train_test_split(data.index, test_size=0.2, random_state=seed)  # Indices of the development records; Indices of the held-out records.
train_idx, test_idx = pd.Index(train_idx), pd.Index(test_idx)
split = data[[point_id, "Catalyst number", "Reference DOI"]].assign(Split="Test")  # Training and held-out partition assignments.
split.loc[train_idx, "Split"] = "Train"
print(f"Records: {len(data)} | Train: {len(train_idx)} | Test: {len(test_idx)}")


def calcination_fill(X_fit):  # Return rounded calcination means for the training rows.
    columns = X_fit.columns.intersection(calc_columns)  # Columns included in the operation.
    means = X_fit[columns].mean()  # Calculated means for imputation or plotting.
    if means.isna().any():
        raise ValueError("A training fold lacks observed calcination values.")
    return np.floor(means + 0.5)


def save_figure(fig, filename, resolution=dpi):  # Save a figure in the results folder and close it.
    settings = {"pil_kwargs": {"compression": "tiff_lzw"}} if filename.endswith(".tif") else {}  
    fig.savefig(output_dir / filename, dpi=resolution, bbox_inches="tight", **settings)
    plt.show()
    plt.close(fig)


def panel_letter(ax, letter, x=-0.16, y=1.04):  # Place a panel identifier on its axes.
    ax.text(x, y, f"({letter})", transform=ax.transAxes, ha="left", va="bottom",
            fontsize=12, fontweight="bold")


In [ ]:


optuna.logging.set_verbosity(optuna.logging.WARNING)
n_splits = 5  # Number of cross-validation folds.
cv_workers = 5  # Cross-validation workers.
folds = list(KFold(n_splits=n_splits, shuffle=True, random_state=seed).split(train_idx))  # Cross-validation splits.
model_defaults = {"objective": "reg:squarederror", "tree_method": "hist", "random_state": seed, "n_jobs": 1}  # Fixed estimator options.


def prediction(model, values, upper=None):  # Predict responses and enforce the configured prediction bounds.
    return np.clip(model.predict(values), 0, upper)


def regression_scores(y, pred):  # Compute regression performance from paired observed and predicted values.
    return {"R2": r2_score(y, pred), "RMSE": np.sqrt(mean_squared_error(y, pred)), "MAE": mean_absolute_error(y, pred)}


def fit_regression(target, columns, trials, upper=None, early_stop=False):  # Tune and fit an XGBoost regressor using the training folds.
    y = data[target].astype(float)  # Observed response.
    if not np.isfinite(y.to_numpy()).all():
        raise ValueError(f"{target} must contain finite observations.")
    X_fit = X.loc[train_idx, columns]  # Predictors used to fit the estimator.
    cv_sets = []  # Prepared cross-validation partitions.
    for fold, (i, j) in enumerate(folds, start=1):  # Cross-validation fold; Loop position; Second loop position.
        X_tr, X_va = X_fit.iloc[i], X_fit.iloc[j]  # Training-fold predictors; Validation-fold predictors.
        fill = calcination_fill(X_tr)  # Rounded imputation values from the fitting data.
        cv_sets.append((fold, X_tr.fillna(fill), y.loc[X_tr.index], X_va.fillna(fill), y.loc[X_va.index]))

    def score_fold(params, fold_data):  # Fit and evaluate the regressor for one validation split.
        fold, X_tr, y_tr, X_va, y_va = fold_data  # Cross-validation fold; Training-fold predictors; Training-fold responses; Validation-fold predictors; Validation-fold responses.
        if early_stop:
            model = XGBRegressor(**model_defaults, **params, n_estimators=1000, early_stopping_rounds=20)  # Estimator for the current task. Limit tuning to 1,000 boosting rounds and use early stopping for Table S3.
            model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            rounds = model.best_iteration + 1  # Number of boosting iterations.
        else:
            model = XGBRegressor(**model_defaults, **params)
            model.fit(X_tr, y_tr)
            rounds = params["n_estimators"]
        return {
            "Fold": fold, "Rounds": rounds, **regression_scores(y_va, prediction(model, X_va, upper)),
            "Train RMSE": regression_scores(y_tr, prediction(model, X_tr, upper))["RMSE"],
        }

    with ThreadPoolExecutor(max_workers=cv_workers) as pool:  # Pool.
        def objective(trial):  # Return cross-validation error for the current Optuna trial.
            if early_stop:
                params = {  # Estimator hyperparameters.
                    "max_depth": trial.suggest_int("max_depth", 2, 5),  # Define the tuned parameter ranges reported in Table S3.
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.1, log=True),
                    "min_child_weight": trial.suggest_int("min_child_weight", 10, 30),
                    "subsample": trial.suggest_float("subsample", 0.5, 0.85),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 0.85),
                    "reg_alpha": trial.suggest_float("reg_alpha", 1e-2, 10, log=True),
                    "reg_lambda": trial.suggest_float("reg_lambda", 1e-2, 50, log=True),
                    "gamma": trial.suggest_float("gamma", 0, 5),
                }
            else:
                params = {
                    "n_estimators": trial.suggest_int("n_estimators", 50, 300),
                    "max_depth": trial.suggest_int("max_depth", 3, 10),
                    "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                    "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                    "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                }
            scores = list(pool.map(lambda s: score_fold(params, s), cv_sets))  # Scores for the current evaluation.
            trial.set_user_attr("fold_scores", scores)
            return float(np.mean([row["RMSE"] for row in scores]))  

        study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=seed))  # Optuna hyperparameter search.
        print(f"Searching {target}: {trials} trials × {n_splits} folds", flush=True)
        study.optimize(objective, n_trials=trials)

    params = study.best_params.copy()  # Estimator hyperparameters.
    fold_scores = pd.DataFrame(study.best_trial.user_attrs["fold_scores"]).assign(Target=target)  
    if early_stop:
        params["n_estimators"] = max(1, int(1.1 * fold_scores["Rounds"].mean()))
    fill = calcination_fill(X_fit)
    fitted_X = X_fit.fillna(fill)  
    test_X = X.loc[test_idx, columns].fillna(fill)  
    model = XGBRegressor(**model_defaults, **params)  # Estimator for the current task.
    model.fit(fitted_X, y.loc[train_idx])

    predicted = pd.Series(index=data.index, dtype=float, name="Predicted")  # Predicted response values.
    metrics = []  # Performance metrics.
    for subset, values in [("Train", fitted_X), ("Test", test_X)]:  # Subset; Values used in the current calculation.
        pred = prediction(model, values, upper)  # Model predictions.
        predicted.loc[values.index] = pred
        metrics.append({"Target": target, "Split": subset, "N": len(pred), **regression_scores(y.loc[values.index], pred)})
    print(f"{target} | trial {study.best_trial.number} | CV RMSE {study.best_value:.4f}", flush=True)
    return model, fitted_X, test_X, fill, params, predicted, pd.DataFrame(metrics), fold_scores, study


def parity_panel(ax, datasets, label, mean_cv_r2=None):  # Plot observed and predicted values on equal axes.
    color_map = {"Train": "#2a9d8f", "Test": "#f4a261"}  # Color map.
    values = np.concatenate([v for _, actual, pred in datasets for v in (actual, pred)])  # Values used in the current calculation. Observed values; Model predictions. Current array or scalar.
    pad = 0.05 * max(np.ptp(values), 1e-9)  # Pad.
    limits = (values.min() - pad, values.max() + pad)  # Limits.
    annotations = []  # Annotations.
    for zorder, (subset, actual, pred) in enumerate(datasets, start=2):  # Zorder; Subset; Observed values; Model predictions.
        score = regression_scores(actual, pred)  # Score.
        ax.scatter(actual, pred, s=40, color=color_map[subset], edgecolor="black", linewidth=0.8,
                   alpha=0.7 if subset == "Train" else 0.8,
                   label=subset, zorder=zorder)
        annotations.append(f"{subset} $R^2$ = {score['R2']:.2f}")
    if mean_cv_r2 is not None:
        annotations.append(f"Mean $R^2$ = {mean_cv_r2:.2f}")
    ax.plot(limits, limits, linestyle="--", dashes=(5, 4), color="black", linewidth=1.5, zorder=1)
    ax.set(xlim=limits, ylim=limits, xlabel=f"Actual {label}", ylabel=f"Predicted {label}")
    ax.set_aspect("equal", adjustable="box")
    for spine in ax.spines.values():  # Spine.
        spine.set_visible(True)
        spine.set_linewidth(1.5)
    ax.tick_params(labelsize=11, direction="in", length=6, width=1.5)
    ax.legend(frameon=False, loc="upper left", fontsize=12)
    ax.text(0.95, 0.05, "\n".join(annotations), transform=ax.transAxes, ha="right", va="bottom",
            multialignment="left", fontsize=12, fontweight="bold",
            bbox={"boxstyle": "round,pad=0.3", "facecolor": "white", "edgecolor": "none", "alpha": 0.8},
            zorder=4)

## Hyperparameter search and prediction

In [ ]:
sty_trials = 100  
sty_unit = r"STY / mg$_{\mathrm{MeOH}}$ h$^{-1}$ g$_{\mathrm{cat}}^{-1}$"  
model, X_train, X_test, fill, params, predicted, metrics, cv_scores, study = fit_regression(  # Estimator for the current task; Development-set predictors; Held-out test predictors; Rounded imputation values from the fitting data; Estimator hyperparameters; Predicted response values; Performance metrics; Cross-validation scores; Optuna hyperparameter search.
    "STY", features, sty_trials, early_stop=True,
)
y = data["STY"]  # Observed response.
fig, ax = plt.subplots(figsize=(5, 5), layout="tight")  # Matplotlib figure; Plot axes.
parity_panel(ax, [
    ("Train", y.loc[train_idx], predicted.loc[train_idx]),
    ("Test", y.loc[test_idx], predicted.loc[test_idx]),
], sty_unit, mean_cv_r2=cv_scores["R2"].mean())
save_figure(fig, "STY_XGB_Square.tif")  # Save Fig. 5, the tuned XGBoost STY parity plot.
display(metrics.round(4))

## SHAP importance and feature dependence

The vertical figure combines the summary with eight dependence panels. Dependence colors indicate relative feature values within each panel, using a shared Low–High scale.

In [ ]:
def tree_explanation(model, X_fit, X_explain):  # Calculate TreeSHAP values for the requested feature rows.
    background = shap.sample(X_fit, min(100, len(X_fit)), random_state=seed)  
    explainer = shap.TreeExplainer(model, data=background, feature_perturbation="interventional")  # TreeSHAP explainer.
    explanation = explainer(X_explain, check_additivity=False)  # TreeSHAP values and their feature data.
    if explanation.values.shape != X_explain.shape:
        raise ValueError("Expected one SHAP value for every sample and feature.")
    return explanation


def shap_panels(explanation, ax_swarm, ax_bar, decimals=3, other_label="Sum of others"):  # Plot SHAP distributions and rank features by mean absolute SHAP.
    mean_abs = np.abs(explanation.values).mean(axis=0)  
    ranked = np.argsort(-mean_abs)  
    top = ranked[:8]  # Highest-ranked entries selected for display.
    top_names = [explanation.feature_names[i] for i in top]  # Top names. Loop position.
    subset = shap.Explanation(  # Subset.
        values=explanation.values[:, top], data=explanation.data[:, top],
        base_values=explanation.base_values, feature_names=top_names,
    )
    plt.sca(ax_swarm)
    np.random.seed(seed)
    shap.plots.beeswarm(subset, show=False, plot_size=None, color=plt.get_cmap("coolwarm"))
    ax_swarm.set_xlabel("SHAP value")
    ax_swarm.spines["bottom"].set_linewidth(1.2)
    ax_swarm.grid(axis="x", linestyle="--", alpha=0.25)
    values = np.r_[mean_abs[top], mean_abs[ranked[8:]].sum()]  # Values used in the current calculation.
    bars = ax_bar.barh(  # Bars.
        top_names + [other_label], values, color=["#1f77b4"] * len(top) + ["#d9d9d9"],
        edgecolor="black", linewidth=0.7, zorder=2,
    )
    ax_bar.invert_yaxis()
    ax_bar.set_xlabel("mean(|SHAP value|)")
    ax_bar.bar_label(bars, fmt=f"%.{decimals}f", padding=3, fontsize=9, fontweight="bold")
    ax_bar.set_xlim(0, max(values.max() * 1.35, 1e-9))
    ax_bar.grid(axis="x", linestyle="--", alpha=0.25)
    return pd.Series(mean_abs, index=explanation.feature_names, name="Mean |SHAP|").sort_values(ascending=False)

In [ ]:
explanation = tree_explanation(model, X_train, X_train)  # TreeSHAP values and their feature data.


def draw_shap_summary(fig, grid):  # Draw the SHAP beeswarm and global feature ranking.
    ax_swarm, ax_bar = fig.add_subplot(grid[0]), fig.add_subplot(grid[1]) 
    ranking = shap_panels(explanation, ax_swarm, ax_bar, decimals=2, other_label="Sum of other features")  
    panel_letter(ax_bar, "b", x=-0.57, y=0.99)
    return ranking

def draw_shap_dependence(fig, grid, ranking):  # Draw SHAP contributions against the eight highest-ranked predictors.
    top = ranking.head(8).index  # Highest-ranked entries selected for display.
    indexes = [X_train.columns.get_loc(feature) for feature in top]  
    limit = max(np.abs(np.quantile(explanation.values[:, indexes], [0.01, 0.99])).max(), 1e-9)  
    norm = mpl.colors.Normalize(0, 1)  
    cmap = plt.get_cmap("viridis")  # Matplotlib color map.
    for i, (feature, index) in enumerate(zip(top, indexes)):  # Loop position; Feature; Current position or index label.
        ax = fig.add_subplot(grid[i // 4, i % 4])  # Plot axes.
        x = X_train[feature].to_numpy()  # Horizontal coordinate or predictor values.
        relative = (x - x.min()) / max(np.ptp(x), 1e-9) 
        ax.scatter(x, explanation.values[:, index], c=relative, cmap=cmap, norm=norm, s=4,
                   alpha=0.85, linewidths=0)
        ax.axhline(0, linestyle="--", color="#d62728", linewidth=0.9)
        ax.set(xlabel=feature, ylabel="SHAP value", ylim=(-limit, limit))
        ax.tick_params(labelsize=7, length=2.5)
        ax.xaxis.label.set_size(8)
        ax.yaxis.label.set_size(8)
        ax.grid(linestyle="--", alpha=0.25)
        panel_letter(ax, chr(99 + i), x=-0.25, y=0.98)
    colorbar = fig.colorbar(mpl.cm.ScalarMappable(norm=norm, cmap=cmap), cax=fig.add_subplot(grid[:, 4]))  
    colorbar.set_ticks([0, 1], labels=["Low", "High"])
    colorbar.set_label("Feature value")
    colorbar.ax.tick_params(labelsize=8)


fig = plt.figure(figsize=(10.2, 5.0))
summary_grid = fig.add_gridspec(1, 2, left=0.24, right=0.97, bottom=0.15, top=0.94,  
                                wspace=0.77, width_ratios=[1.18, 1])
shap_rank = draw_shap_summary(fig, summary_grid)  # Features ranked by mean absolute SHAP.
save_figure(fig, "SHAP_summary.tif")  # Save Fig. 6a (SHAP beeswarm) and Fig. 6b (mean absolute SHAP).

fig = plt.figure(figsize=(10.2, 4.1))
dep_grid = fig.add_gridspec(2, 5, left=0.08, right=0.94, bottom=0.12, top=0.95,  # Dep grid.
                            wspace=0.45, hspace=0.42, width_ratios=[1, 1, 1, 1, 0.17])
draw_shap_dependence(fig, dep_grid, shap_rank)
save_figure(fig, "SHAP_dependence.tif")  # Save Fig. 6c–j, SHAP dependence panels in feature-rank order.

fig = plt.figure(figsize=(10.2, 9.0))
sections = fig.add_gridspec(2, 2, left=0.08, right=0.95, top=0.96, bottom=0.065,  # Sections.
                            height_ratios=[1.24, 1], width_ratios=[0.16, 0.84], hspace=0.24)
summary_grid = sections[0, 1].subgridspec(1, 2, wspace=0.77, width_ratios=[1.18, 1])
dep_grid = sections[1, :].subgridspec(2, 5, wspace=0.45, hspace=0.42,
                                      width_ratios=[1, 1, 1, 1, 0.17])
draw_shap_summary(fig, summary_grid)
draw_shap_dependence(fig, dep_grid, shap_rank)
save_figure(fig, "Final_Combined_Vertical.tif", resolution=1200)  # Save Fig. 6a–j, the combined STY SHAP summary and dependence plots.

## Permutation importance on the held-out test set

Individual columns and feature groups are permuted over the same test records. Each group uses one row permutation for all its member columns. The score is the decrease in test R².

In [ ]:
feature_groups = {  
    "Catalyst composition": features[:9],
    "Pretreatment": features[9:12],
    "Reaction conditions": features[12:16],
    "Characterization": features[16:],
}
colors = ["#08306B", "#67000D", "#D94801", "#6A51A3"]  
repeats = 50  
rng = np.random.default_rng(seed)  
permutations = [rng.permutation(len(X_test)) for _ in range(repeats)] 
baseline_r2 = r2_score(y.loc[test_idx], predicted.loc[test_idx])  # Baseline R-squared.


def permutation_scores(columns):  # Measure the score decrease after permuting selected features.
    drops = []  
    original = X_test[columns].to_numpy()  
    for order in permutations:  
        sample = X_test.copy()  
        sample[columns] = original[order]
        drops.append(baseline_r2 - r2_score(y.loc[test_idx], prediction(model, sample)))
    return {"Mean R2 drop": np.mean(drops), "SD": np.std(drops, ddof=1)}


individual_pfi = pd.DataFrame([  # Individual permutation importance.
    {"Group": group, "Feature": feature, **permutation_scores([feature])}
    for group, columns in feature_groups.items() for feature in columns 
])
grouped_pfi = pd.DataFrame([  # Grouped permutation importance.
    {"Group": group, **permutation_scores(columns)} for group, columns in feature_groups.items() 
])
colors_by_group = dict(zip(feature_groups, colors))  


def draw_pfi(ax, names, values, bar_colors, letter, label, decimals):  # Draw permutation scores with their spread.
    bars = ax.bar(names, values, color=bar_colors, edgecolor="black", linewidth=0.8, zorder=3)  
    ax.bar_label(bars, fmt=f"%.{decimals}f", padding=2, fontsize=8)
    ax.axhline(0, color="0.35", linewidth=0.7)
    ax.margins(y=0.16)
    plt.setp(ax.get_xticklabels(), rotation=27, ha="right", fontsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.25)
    ax.set(xlabel=label, ylabel=r"$R^2$ score drop")
    panel_letter(ax, letter, x=-0.10, y=1.02)


def draw_individual_pfi(axes):  # Draw permutation importance for the features of each group.
    for i, (ax, (group, rows)) in enumerate(zip(axes, individual_pfi.groupby("Group", sort=False))):  # Loop position; Plot axes; Group; Rows accumulated for the result table.
        ranked = rows.sort_values("Mean R2 drop", ascending=False)  # Ranked.
        names, values = ranked["Feature"].tolist(), ranked["Mean R2 drop"].to_numpy()  # Names; Values used in the current calculation.
        if group == "Catalyst composition":
            names, values = names[:3] + ["Others (sum)"], np.r_[values[:3], values[3:].sum()]
        draw_pfi(ax, names, values, colors_by_group[group], chr(97+i), group, 4)


def draw_grouped_pfi(ax):  # Draw permutation importance for entire feature groups.
    ranked = grouped_pfi.sort_values("Mean R2 drop", ascending=False)  
    draw_pfi(ax, ranked["Group"], ranked["Mean R2 drop"], ranked["Group"].map(colors_by_group),
             "e", "Grouped Permutation Importance", 2)


fig, axes = plt.subplots(2, 2, figsize=(9, 8), layout="constrained")  # Plot axes arranged by panel.
draw_individual_pfi(axes.flat)
save_figure(fig, "Permutation_Importance_Features.tif")  # Save Fig. S2a–d, permutation importance within each feature group.
fig, ax = plt.subplots(figsize=(7, 5), layout="constrained")
draw_grouped_pfi(ax)
save_figure(fig, "Permutation_Importance_Groups.tif")  # Save Fig. S2e, permutation importance for entire groups.
fig = plt.figure(figsize=(9, 12), layout="constrained")
layout = fig.add_gridspec(3, 4)  # Layout.
draw_individual_pfi([fig.add_subplot(layout[row, col:col+2]) for row in range(2) for col in (0, 2)])  # Current row. Col.
draw_grouped_pfi(fig.add_subplot(layout[2, 1:3]))
save_figure(fig, "Permutation_Importance_Combined.tif")  # Save Fig. S2a–e as a combined figure.

## One- and two-feature partial dependence

The curves average raw XGBoost predictions over training records. The two-feature grids use the central 5th–95th percentile range for continuous features, with observed training points shown inside each panel.

In [ ]:

pdp_features = ["GHSV", "P", "T", "H2/CO2", "SBET", "SACu", "wt% Cu", "wt% Support1", "wt% Support2"]  # Partial dependence features.
pdp_1d, pdp_2d = [], []  # Single-feature partial dependence values; Two-feature partial dependence values.
for feature in pdp_features:  
    result = partial_dependence(  
        model, X_train, [feature], grid_resolution=40, percentiles=(0, 1), method="brute",
    )
    grid, average = result["grid_values"][0], result["average"][0]  
    pdp_1d.append(pd.DataFrame({"Feature": feature, "Value": grid, "Mean prediction": average}))
curves = {frame["Feature"].iat[0]: frame for frame in pdp_1d}  # Curves. Data frame for the current partition.


def draw_pdp_curve(ax, feature):  # Draw the mean prediction across a single-feature grid.
    curve = curves[feature] 
    ax.plot(curve["Value"], curve["Mean prediction"], color="green", linewidth=1.5)
    ax.set(xlabel=feature, ylabel="Partial dependence")
    ax.tick_params(direction="out", length=3, labelsize=9)


fig, axes = plt.subplots(2, 4, figsize=(13, 5.8), layout="constrained")
panel_order = ["GHSV", "T", "P", "H2/CO2", "SACu", "SBET", "wt% Cu", "wt% Support1"]  # Panel order.
for i, (ax, feature) in enumerate(zip(axes.flat, panel_order)):  
    draw_pdp_curve(ax, feature)
    panel_letter(ax, chr(97+i), x=-0.20, y=1.02)
save_figure(fig, "PDP_1D.tif")  # Save Fig. S3a–h in the specified single-feature panel order.
fig, ax = plt.subplots(figsize=(4, 3.3), layout="constrained")
draw_pdp_curve(ax, "wt% Support2")
save_figure(fig, "PDP_wt_pct_Support2.tif")

pairs = list(combinations(shap_rank.head(4).index, 2))  
fig, axes = plt.subplots(2, 3, figsize=(14, 8), layout="constrained")
cmap = mpl.colors.LinearSegmentedColormap.from_list(  # Matplotlib color map.
    "interaction", ["#d00000", "#ffba08", "#3f88c5", "#032b43", "#136f63", "#116359", "#0c413b"],
)
for i, (ax, (first, second)) in enumerate(zip(axes.flat, pairs)):  # First item in the pair; Second item in the pair.
    result = partial_dependence(model, X_train, [first, second], grid_resolution=22, method="brute")
    gx, gy = result["grid_values"]  
    average = result["average"][0]
    if min(len(gx), len(gy)) > 1 and np.ptp(average) > 0:
        levels = np.linspace(average.min(), average.max(), 8)
        image = ax.contourf(gx, gy, average.T, levels=levels, cmap=cmap)  
        lines = ax.contour(gx, gy, average.T, levels=levels[1:-1], colors="white", linewidths=0.55)  
        ax.clabel(lines, inline=True, fontsize=7, fmt="%.1f", colors="black")
    else:
        image = ax.pcolormesh(gx, gy, average.T, shading="nearest", cmap=cmap)
    fig.colorbar(image, ax=ax, pad=0.015, fraction=0.037, format="%.1f")
    ax.set(xlabel=first, ylabel=second, title=f"Partial Dependence of {first} and {second}")
    ax.title.set_size(10)
    panel_letter(ax, chr(97+i), x=0.035, y=0.92)
    for feature, axis_handle in [(first, ax.xaxis), (second, ax.yaxis)]:  
        if X_train[feature].max() > 1000:
            axis_handle.set_major_formatter(mpl.ticker.StrMethodFormatter("{x:,.0f}"))
    grid_x, grid_y = np.meshgrid(gx, gy, indexing="ij")  
    pdp_2d.append(pd.DataFrame({
        "Feature X": first, "Feature Y": second, "X": grid_x.ravel(), "Y": grid_y.ravel(),
        "Mean prediction": average.ravel(),
    }))
save_figure(fig, "PDP_2D_Interactions.tif")  # Save Fig. S4a–f, partial dependence for the pairs of the four leading SHAP features.

## Export STY results

In [ ]:
predictions = split.assign(STY=y, Predicted=predicted, Residual=predicted - y)  # Record-level predictions.
high_sty = predictions.loc[predictions["STY"].gt(1000)]  
high_sty_summary = high_sty.groupby("Split")["Residual"].agg(N="size", Bias="mean")  
high_sty_summary["MAE"] = high_sty["Residual"].abs().groupby(high_sty["Split"]).mean()
cv_summary = cv_scores[["R2", "RMSE", "MAE"]].agg(["mean", "std"])  # Cross-validation summary.
with pd.ExcelWriter(output_dir / "STY_results.xlsx", engine="openpyxl") as writer:  # Excel writer for the result workbook.
    metrics.to_excel(writer, sheet_name="metrics", index=False)
    cv_scores.to_excel(writer, sheet_name="cv_folds", index=False)
    cv_summary.to_excel(writer, sheet_name="cv_summary")
    pd.Series(params, name="Value").to_excel(writer, sheet_name="best_params")  # Save the selected values reported in Table S3.
    study.trials_dataframe(attrs=("number", "value", "params", "state")).to_excel(writer, sheet_name="optuna_trials", index=False)
    predictions.to_excel(writer, sheet_name="predictions", index=False)
    high_sty_summary.to_excel(writer, sheet_name="STY_over_1000")
    shap_rank.to_excel(writer, sheet_name="shap_importance")
    individual_pfi.to_excel(writer, sheet_name="permutation_features", index=False)
    grouped_pfi.to_excel(writer, sheet_name="permutation_groups", index=False)
    pd.concat(pdp_1d).to_excel(writer, sheet_name="pdp_1d", index=False)
    pd.concat(pdp_2d).to_excel(writer, sheet_name="pdp_2d", index=False)
